# 02. Preprocessing — 데이터 전처리

## Google Colab 환경 설정

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    !git clone https://github.com/Lunarian0928/youth_bio_global_portfolio.git
    %cd youth_bio_global_portfolio/oct_retinal_segmentation

    !pip install -q segmentation-models-pytorch albumentations


## Imports

In [ ]:
import os
import re
import sys

sys.path.append("..")

import albumentations as A
import pandas as pd
from sklearn.model_selection import train_test_split

from src.dataset import DEFAULT_DISEASES, OCTRetinalDataset


## Config

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/data/OCT5k" if IN_COLAB else "../data/OCT5k"
# Colab은 %cd로 oct_retinal_segmentation/ 폴더 자체가 cwd가 되고,
# 로컬 Jupyter는 notebooks/ 폴더가 cwd이므로 "../" 깊이가 다르다.
SPLITS_DIR = "data/splits" if IN_COLAB else "../data/splits"
GRADING = 1
RANDOM_SEED = 42


## 원본 데이터 로드

In [ ]:
dataset = OCTRetinalDataset(root_dir=DATA_ROOT, grading=GRADING)
print("총 샘플 수:", len(dataset))


## Resize & Normalize

In [ ]:
# OCT5k 다운로드 준비 단계에서 이미 모든 이미지/마스크가 512x512로 리사이즈되어 있고,
# OCTRetinalDataset.__getitem__에서 이미지를 0~1로 정규화한다. 여기서는 그 결과만 확인한다.
image, mask = dataset[0]
print("image shape/dtype:", image.shape, image.dtype, "min/max:", image.min().item(), image.max().item())
print("mask shape/dtype:", mask.shape, mask.dtype, "unique:", mask.unique())


## Augmentation 정의

In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
])

val_transform = None  # 검증/테스트는 augmentation 없이 그대로 사용


## Train / Validation / Test Split

In [ ]:
# 같은 환자(E2E 파일)의 단면 이미지가 train/val/test에 동시에 섞이면 데이터 누수가 생기므로
# 환자 단위로 그룹을 묶어서 분할한다.
# 경로는 절대경로로 저장해서, 이 csv를 어느 작업 디렉토리에서 읽어도 깨지지 않게 한다.
PATIENT_PATTERN = re.compile(r"([^/\\]+\.E2E)")


def get_disease(image_path):
    for disease in DEFAULT_DISEASES:
        if f"{os.sep}{disease}{os.sep}" in image_path:
            return disease
    return None


def get_patient_id(image_path):
    match = PATIENT_PATTERN.search(image_path)
    return match.group(1) if match else image_path


records = pd.DataFrame(
    [
        {
            "image_path": os.path.abspath(image_path),
            "mask_path": os.path.abspath(mask_path),
            "disease": get_disease(image_path),
            "patient_id": get_patient_id(image_path),
        }
        for image_path, mask_path in dataset.samples
    ]
)

patients = records[["patient_id", "disease"]].drop_duplicates()

train_patients, temp_patients = train_test_split(
    patients, test_size=0.3, stratify=patients["disease"], random_state=RANDOM_SEED
)
val_patients, test_patients = train_test_split(
    temp_patients, test_size=0.5, stratify=temp_patients["disease"], random_state=RANDOM_SEED
)

train_df = records[records["patient_id"].isin(train_patients["patient_id"])]
val_df = records[records["patient_id"].isin(val_patients["patient_id"])]
test_df = records[records["patient_id"].isin(test_patients["patient_id"])]

print(f"train: {len(train_df)} (환자 {len(train_patients)})")
print(f"val:   {len(val_df)} (환자 {len(val_patients)})")
print(f"test:  {len(test_df)} (환자 {len(test_patients)})")


## K-Fold Cross-Validation (환자 단위)

In [ ]:
# 환자가 60명뿐이라 고정 val/test(9명)만으로는 지표가 fold마다 크게 흔들릴 수 있다.
# test_patients는 그대로 최종 평가용으로 떼어두고, 나머지(train+val 환자)를 K-Fold로 나눠
# "고정 split"과 "K-Fold 평균"을 같은 test set 기준으로 비교할 수 있게 한다.
from sklearn.model_selection import StratifiedGroupKFold

N_FOLDS = 5

cv_pool = records[~records["patient_id"].isin(test_patients["patient_id"])].reset_index(drop=True)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_assignments = {}  # patient_id -> fold index
for fold_idx, (_, val_idx) in enumerate(
    sgkf.split(cv_pool, y=cv_pool["disease"], groups=cv_pool["patient_id"])
):
    for patient_id in cv_pool.iloc[val_idx]["patient_id"].unique():
        fold_assignments[patient_id] = fold_idx

cv_pool["fold"] = cv_pool["patient_id"].map(fold_assignments)

for fold_idx in range(N_FOLDS):
    fold_val = cv_pool[cv_pool["fold"] == fold_idx]
    fold_train = cv_pool[cv_pool["fold"] != fold_idx]
    print(
        f"fold {fold_idx}: train {len(fold_train)} (환자 {fold_train['patient_id'].nunique()}), "
        f"val {len(fold_val)} (환자 {fold_val['patient_id'].nunique()})"
    )


## 전처리된 데이터 저장

In [ ]:
os.makedirs(SPLITS_DIR, exist_ok=True)

# 고정 split (빠른 개발/디버깅용)
train_df[["image_path", "mask_path"]].to_csv(os.path.join(SPLITS_DIR, "train.csv"), index=False)
val_df[["image_path", "mask_path"]].to_csv(os.path.join(SPLITS_DIR, "val.csv"), index=False)
test_df[["image_path", "mask_path"]].to_csv(os.path.join(SPLITS_DIR, "test.csv"), index=False)

# K-Fold split (최종 성능 보고용) — test.csv는 고정 split과 동일하게 공유
for fold_idx in range(N_FOLDS):
    fold_val = cv_pool[cv_pool["fold"] == fold_idx]
    fold_train = cv_pool[cv_pool["fold"] != fold_idx]
    fold_train[["image_path", "mask_path"]].to_csv(
        os.path.join(SPLITS_DIR, f"fold{fold_idx}_train.csv"), index=False
    )
    fold_val[["image_path", "mask_path"]].to_csv(
        os.path.join(SPLITS_DIR, f"fold{fold_idx}_val.csv"), index=False
    )

print(f"고정 split: {SPLITS_DIR}/{{train,val,test}}.csv")
print(f"K-Fold split: {SPLITS_DIR}/fold{{0..{N_FOLDS - 1}}}_{{train,val}}.csv (test.csv 공유)")
